In [2]:
# fix_3_reranking.py
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
reranker        = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Simulate a query and 10 candidates from vector DB
query = "How do I get a refund for a duplicate charge?"

candidate_chunks = [
    "To request a refund, contact billing support with your order ID.",           # relevant
    "Refunds for duplicate charges are processed within 3-5 business days.",      # highly relevant
    "Our return policy covers physical items within 30 days of purchase.",        # off-topic
    "If you see an unexpected charge, review your subscription plan first.",      # partially relevant
    "Double billing issues are escalated to the finance team automatically.",     # relevant
    "You can update your payment method from the account settings page.",         # off-topic
    "Subscription upgrades take effect at the start of the next billing cycle.",  # off-topic
    "Contact support if your refund has not arrived after 7 business days.",      # relevant
    "Billing disputes can be submitted through the Help Center portal.",          # relevant
    "Account charges are visible under the Billing tab in your dashboard.",       # off-topic
]

In [4]:
# Step 1: Embedding similarity ranking (naive approach)
q_emb = embedding_model.encode(query)
emb_scores = [
    np.dot(q_emb, embedding_model.encode(chunk)) /
    (np.linalg.norm(q_emb) * np.linalg.norm(embedding_model.encode(chunk)))
    for chunk in candidate_chunks
]
emb_ranked = sorted(zip(emb_scores, candidate_chunks), reverse=True)

In [5]:
# Step 2: Cross-encoder reranking
pairs  = [(query, chunk) for chunk in candidate_chunks]
scores = reranker.predict(pairs)
reranked = sorted(zip(scores, candidate_chunks), reverse=True)

In [6]:
# Compare
print("TOP 5 — Embedding Similarity (before reranking):")
print("-" * 65)
for score, chunk in emb_ranked[:5]:
    print(f"  [{score:.4f}] {chunk}")

print("\nTOP 5 — Cross-Encoder Reranking (after reranking):")
print("-" * 65)
for score, chunk in reranked[:5]:
    print(f"  [{score:.3f}] {chunk}")

TOP 5 — Embedding Similarity (before reranking):
-----------------------------------------------------------------
  [0.8104] Refunds for duplicate charges are processed within 3-5 business days.
  [0.7639] To request a refund, contact billing support with your order ID.
  [0.6808] If you see an unexpected charge, review your subscription plan first.
  [0.6487] Contact support if your refund has not arrived after 7 business days.
  [0.6466] Our return policy covers physical items within 30 days of purchase.

TOP 5 — Cross-Encoder Reranking (after reranking):
-----------------------------------------------------------------
  [6.302] Refunds for duplicate charges are processed within 3-5 business days.
  [-1.100] To request a refund, contact billing support with your order ID.
  [-6.797] Contact support if your refund has not arrived after 7 business days.
  [-10.491] If you see an unexpected charge, review your subscription plan first.
  [-10.623] Our return policy covers physical item